# Continuous ASL Foundation Model Training (Keras 3 + JAX on Cloud TPU v5e-1)
This notebook trains the Continuous ASL Foundation Model using **Keras 3 with JAX Backend** on Google Colab **TPU v5e-1** (Single TensorCore, 16 GB HBM).

- **Backend**: `KERAS_BACKEND=jax` with `@jax.jit` fused XLA execution
- **Architecture**: MobileConformer Visual Landmark Encoder + ASL Transformer Decoder
- **Precision**: `mixed_bfloat16` hardware acceleration
- **Target Throughput**: ~250+ samples/s (pure single-core systolic MXU saturation)
- **Memory Configuration**: `PJRT_ALLOCATOR_FRACTION=0.90`, zero graph recompilations

### Step 1: Configure JAX Backend & Verify TPU v5e-1 Hardware

In [ ]:
# 1. Ensure TPU v5e environment variables
import os
os.environ["KERAS_BACKEND"] = "jax"
os.environ["PJRT_DEVICE"] = "TPU"
os.environ["PJRT_ALLOCATOR_FRACTION"] = "0.90"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.90"

# 2. Verify JAX and Keras 3 on Cloud TPU
!pip install -q kagglehub
import jax
import keras

print("[+] JAX Devices:", jax.devices())
print(f"[+] Keras version: {keras.__version__}, Backend: {keras.backend.backend()}")

# 3. Enable mixed precision policy for TPU systolic MXUs
keras.mixed_precision.set_global_policy("mixed_bfloat16")
print("[+] Global mixed precision policy:", keras.mixed_precision.global_policy())

### Step 2: Setup Repository & Download Datasets

In [ ]:
import os
import kagglehub

# Clone repository if not already present
if not os.path.exists("/content/netmaui-singlanguage"):
    print("[*] Cloning repository...")
    !git clone https://github.com/Codering2012/netmaui-singlanguage.git /content/netmaui-singlanguage

%cd /content/netmaui-singlanguage

print("[*] Downloading ASL dataset from Kaggle...")
asl_data_path = kagglehub.dataset_download("tranquocbao2012/frakenstein-asl-final-version")
print("[+] ASL Dataset downloaded to:", asl_data_path)

print("[*] Downloading ASLG-PC12 and Wikimedia data...")
aslg_path = kagglehub.dataset_download("thedevastator/unlock-the-power-of-english-asl-with-aslg-pc12-c")
kdwd_path = kagglehub.dataset_download("kenshoresearch/kensho-derived-wikimedia-data")

print("[*] Downloading Pre-trained Phase 1 checkpoint...")
phase1_model_path = kagglehub.model_download("muddragonmike/pretrain-bidirectional-asl-english/pytorch/default/1")
print("[+] Pretrained weights downloaded to:", phase1_model_path)

### Step 3: Verify High-Throughput Hardware Targets on TPU v5e-1
Targets:
- **Phase 1 (CLM)**: 1,000 - 1,200 samples/s
- **Phase 2 & 3**: >= 300 seq/s
Config: `--d-model 512 --max-len 384 --english-max-len 128 --chicago-max-len 128`

In [ ]:
# 1. Run Comprehensive TPU v5e-1 Benchmark Suite (Phase 1, Phase 2, Phase 3)
!python3 train_keras/benchmark_tpu_speed.py

# 2. Run Real Data Phase 1 Benchmark streaming from ASLG-PC12 dataset
!python3 train_keras/benchmark_phase1_real_data.py

### Step 4: Launch Continuous ASL Foundation Model Training on TPU v5e-1

In [ ]:
# Path resolution
data_dir = os.path.join(asl_data_path, "asl_dataset", "asl_preprocessed_phase1")
aslg_csv = os.path.join(aslg_path, "train.csv")
kdwd_dir = kdwd_path
phase1_ckpt = os.path.join(phase1_model_path, "asl_llm_200")

os.makedirs("/content/checkpoints", exist_ok=True)

# Launch training with Keras 3 JAX backend on TPU v5e-1
!python3 train_keras/train_tpu_keras.py \
  --data-dir "$data_dir" \
  --kdwd-dir "$kdwd_dir" \
  --aslg-csv "$aslg_csv" \
  --phase1-checkpoint "$phase1_ckpt" \
  --save-dir "/content/checkpoints" \
  --precision mixed_bfloat16 \
  --batch-size 256 \
  --epochs 100 \
  --max-len 384 \
  --english-max-len 128 \
  --chicago-max-len 128 \
  --text-max-len 128 \
  --d-model 512 \
  --nhead 4 \
  --kv-heads 2 \
  --num-layers 4